**RAG Capstone Project**

Installing libraries

In [ ]:
pip install pymupdf
pip install langchain
pip install langchain-text-splitters
pip install faiss-cpu chromadb sentence-transformers openai
pip install -U langchain-google-genai
pip install streamlit

In [ ]:
import json
import re
import time
import fitz
from pathlib import Path
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

#### Document Parsing

In [ ]:
INPUT_FOLDER = "/content/RAG_Capstone_data"
OUTPUT_FILE = "parsed_documents.json"

Cleaning the files

In [ ]:
def clean_text(text: str) -> str:
    """Remove extra spaces and blank lines."""
    text = re.sub(r"[ \t]+", " ", text)   # collapse multiple spaces/tabs into one
    text = re.sub(r"\n{2,}", "\n", text)  # collapse multiple blank lines into one
    return text.strip()

Converting the files into Json

In [ ]:
def main():

    pdf_folder = Path(INPUT_FOLDER)
    pdf_files = sorted(pdf_folder.glob("*.pdf"))

    if not pdf_files:
        print(f"No PDFs found in '{INPUT_FOLDER}'. Check the folder name.")
        return

    all_documents = []

    for pdf_path in pdf_files:
        start = time.time()
        print(f"Reading: {pdf_path.name}")

        doc = pymupdf.open(pdf_path)
        num_pages = len(doc)

        # Pull the text out of every page and join it into one string
        page_texts = [page.get_text("text") for page in doc]
        raw_text = "\n".join(page_texts)
        cleaned_text = clean_text(raw_text)
        doc.close()
        cleaned_text = clean_text(raw_text)

        all_documents.append({
            "source_file": pdf_path.name,
            "text": cleaned_text,
            "char_count": len(cleaned_text),
        })

        elapsed = time.time() - start
        print(f"  -> {num_pages} pages, {len(cleaned_text)} characters, "
              f"took {elapsed:.1f}s")


     # Save everything into one JSON file
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_documents, f, ensure_ascii=False, indent=2)

    print(f"\nDone! Saved {len(all_documents)} documents to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
import json

OUTPUT_FILE = "parsed_documents.json" # Added for robustness

# Load the parsed documents
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    parsed_documents = json.load(f)

print(f"Loaded {len(parsed_documents)} documents from {OUTPUT_FILE}")

#### Text Chunking

In [ ]:
def pdf_to_json(path, company="Unknown"):
    doc = fitz.open(path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    output = {"chunks": []}

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")

        # Detect section (simple heuristic)
        section = "General"
        for line in text.split("\n"):
            if (line.isupper() and len(line) < 80) or re.match(r"^\d+\.", line):
                section = line.strip()
                break

        # Chunk the page content
        chunks = splitter.split_text(text)
        for i, chunk in enumerate(chunks):
            output["chunks"].append({
                "text": chunk,
                "metadata": {
                    "company": company,
                    "section": section,
                    "page": page_num,
                    "chunk_id": f"{company}_{page_num}_{i}"
                }
            })
    return output

# Example usage

MANUAL_COMPANY_OVERRIDES = {
    "/content/RAG_Capstone_data/GM 2025 Climate-Related Disclosures Report.pdf": "General Motors",
    "/content/RAG_Capstone_data/alphabet-2024-cdp-climate-change-response.pdf": "Alphabet",
    "/content/RAG_Capstone_data/2024-environmental-update.pdf": "Coca-Cola",
    "/content/RAG_Capstone_data/IKEA_Sustainability_Report_FY_24_2025_01_27_2c35989733.pdf": "IKEA",
     "/content/RAG_Capstone_data/251106-our-net-zero-transition-plan.pdf": "HSBC",
    "/content/RAG_Capstone_data/2026-Microsoft-Environmental-Sustainability-Report-PDF.pdf": "Microsoft",
    "/content/RAG_Capstone_data/google-2024-environmental-report.pdf": "Google",
      "/content/RAG_Capstone_data/unilever-annual-report-and-accounts-2024.pdf": "Unilever",
    "/content/RAG_Capstone_data/Apple_Environmental_Progress_Report_2024.pdf": "Apple",
    "/content/RAG_Capstone_data/2024-Responsible-Business-Report.pdf": "Verizon"
}
all_documents = []
for file_name, company_name in MANUAL_COMPANY_OVERRIDES.items():
     parsed_json = pdf_to_json(file_name, company=company_name)
     # Wrap each company’s chunks into a structured object
     all_documents.append({
        "company": company_name,
        "file": file_name,
        "chunks": parsed_json
       })

# Save all companies together
with open("/content/parsed_documents.json", "w", encoding="utf-8") as f:
    json.dump(all_documents, f, indent=2)

#### Embedding Generation

In [ ]:
import json

with open("/content/parsed_documents.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Initialize an empty list to store all the individual text chunks from all documents
all_chunks = []

# Iterate through each document (company) in the loaded data
for company_doc in data:
    # Each company_doc is expected to be a dictionary like:
    # {"company": ..., "file": ..., "chunks": {"chunks": [...]}}
    # accessing the inner "chunks" list from each document
    if "chunks" in company_doc and isinstance(company_doc["chunks"], dict) and "chunks" in company_doc["chunks"]:
        all_chunks.extend(company_doc["chunks"]["chunks"])


In [ ]:
from sentence_transformers import SentenceTransformer

bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
bge_embeddings = [bge_model.encode(chunk["text"]) for chunk in all_chunks]

In [ ]:
import faiss
import numpy as np

# Example with BGE embeddings
embedding_matrix = np.array(bge_embeddings).astype("float32")

index = faiss.IndexFlatL2(embedding_matrix.shape[1])
index.add(embedding_matrix)

# Save FAISS index
faiss.write_index(index, "company_chunks.index")

#### Retrieval System

In [ ]:
def retrieve(query, chunks, model, index, k=3):
    # Encode the query into an embedding
    query_embedding = model.encode(query).astype("float32")

    # Search FAISS index
    D, I = index.search(np.array([query_embedding]), k)

    # Collect results
    results = []
    for idx, dist in zip(I[0], D[0]):
        if idx < len(chunks):  # safety check
            results.append({
                "text": chunks[idx]["text"],
                "metadata": chunks[idx]["metadata"],
                "distance": float(dist)
            })
    return results


In [ ]:
print('--- Testing the retrieve function ---')

sample_queries = [
    "What are Apple’s goals for carbon neutrality?",
    "How does Unilever manage plastic waste?",
    "What steps has Coca-Cola taken to reduce water usage?"
]

for query in sample_queries:
    print(f"\nQuery: {query}")
    retrieved_docs = retrieve(query, all_chunks, bge_model, index, k=1)
    for i, doc in enumerate(retrieved_docs):
        print(f"  Retrieved Document {i+1} (Company: {doc['metadata']['company']}, Page: {doc['metadata']['page']}, Distance: {doc['distance']:.4f}):")
        print(f"    {doc['text'][:200]}...") # Print first 200 characters of the text

#### Define and Test the RAG Chain

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Get the API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Configure the API key
genai.configure(api_key=GOOGLE_API_KEY)


print("Listing available Generative AI models that support generateContent:")
selected_model = None
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(f"  Model: {m.name}, Description: {m.description}")
    # Prioritize 'models/gemini-flash-latest' if available
    if m.name == "models/gemini-flash-latest":
        selected_model = m.name

if selected_model:
    print(f"\nSelected model for RAG: {selected_model}")
else:
    # Fallback to a common model if nothing was explicitly selected
    selected_model = "models/gemini-flash-latest" # Defaulting to the most common flash model
    print(f"\nNo preferred model found. Defaulting to {selected_model}. Please verify if this model is supported in your region.")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

# Initialize the LLM with the selected model
llm = ChatGoogleGenerativeAI(model=selected_model, google_api_key=GOOGLE_API_KEY, temperature=0.2)

# Define the prompt template for RAG
rag_prompt_template = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template)

# Define a function to format the retrieved documents into a single string
def format_docs(docs):
    # docs will be a list of dicts, each with a 'text' key from our retrieve function
    return "\n\n".join(doc['text'] for doc in docs)

# Define the RAG chain
# The 'retrieve' function takes the query, all_chunks, bge_model, and index
# and returns the relevant chunks. These chunks are then formatted and passed as context.
rag_chain = (
    {"context": lambda x: format_docs(retrieve(x["question"], all_chunks, bge_model, index)), "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("RAG chain defined successfully!")

#### LLM-as-a-Judge Evaluation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define a prompt for the LLM judge
judge_prompt_template = """You are an impartial judge. Your task is to evaluate the provided answer based on the given question and context. You should rate the answer on a scale of 1 to 5 for relevance, faithfulness, and conciseness.

Criteria:
-   **Relevance (1-5):** How well does the answer directly address the question?
-   **Faithfulness (1-5):** Is the answer fully supported by the provided context? Does it avoid hallucination?
-   **Conciseness (1-5):** Is the answer brief and to the point, adhering to a maximum of three sentences?

Provide your ratings and a brief explanation for each.

Question: {question}
Context: {context}
Answer: {answer}

Evaluation:
"""

judge_prompt = ChatPromptTemplate.from_template(judge_prompt_template)

# Create an evaluation chain
eval_chain = (
    {
        "question": RunnablePassthrough(),
        "context": lambda x: format_docs(retrieve(x["question"], all_chunks, bge_model, index)),
        "answer": lambda x: rag_chain.invoke({"question": x["question"]})
    }
    | judge_prompt
    | llm # Using the same LLM for judging
    | StrOutputParser()
)

print("LLM-as-a-Judge evaluation chain defined successfully!")
print(f"LLM temperature set to: {llm.temperature}")

In [ ]:
print('--- Running LLM-as-a-Judge Evaluation ---')

queries_for_evaluation = [
    "What are Apple’s goals for carbon neutrality?",
    "How does Unilever manage plastic waste?",
    "What steps has Coca-Cola taken to reduce water usage?"
]

for query in queries_for_evaluation:
    print(f"\n--- Evaluating for Question: {query} ---")
    try:
        print("Retrieving documents...")
        # Context retrieval is done inside the eval_chain as part of 'context' lambda

        print("Generating RAG answer...")
        # The RAG answer is generated inside the eval_chain as part of 'answer' lambda

        print("Invoking LLM judge...")
        evaluation_result = eval_chain.invoke({"question": query})
        print(evaluation_result)
    except Exception as e:
        print(f"Error during evaluation for '{query}': {e}")

In [ ]:
!npm install -g localtunnel

#### Deploying in Streamlit

In [ ]:
%%writefile /content/app.py

import streamlit as st
import numpy as np
import faiss
import json
import os
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import google.generativeai as genai

# Load pre-trained models and data (must be consistent with previous notebook steps)
# Load the parsed documents
with open("/content/parsed_documents.json", "r", encoding="utf-8") as f:
    data = json.load(f)

all_chunks = []
for company_doc in data:
    if "chunks" in company_doc and isinstance(company_doc["chunks"], dict) and "chunks" in company_doc["chunks"]:
        all_chunks.extend(company_doc["chunks"]["chunks"])

# Load the SentenceTransformer model
bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

# Load the FAISS index
index = faiss.read_index("company_chunks.index")

# Define the retrieve function (copied from previous cells)
def retrieve(query, chunks, model, index, k=1):
    # Encode the query into an embedding
    query_embedding = model.encode(query).astype("float32")

    # Search FAISS index
    D, I = index.search(np.array([query_embedding]), k)

    # Collect results
    results = []
    for idx, dist in zip(I[0], D[0]):
        if idx < len(chunks):  # safety check
            results.append({
                "text": chunks[idx]["text"],
                "metadata": chunks[idx]["metadata"],
                "distance": float(dist)
            })
    return results

# Initialize the LLM and RAG chain (copied from previous cells)
# Get the API key from environment variables
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

if not GOOGLE_API_KEY:
    st.error("Google API Key not found. Please set the GOOGLE_API_KEY environment variable.")
    st.stop()

genai.configure(api_key=GOOGLE_API_KEY)

selected_model = "models/gemini-flash-latest" # Assuming this was the selected model from previous run
llm = ChatGoogleGenerativeAI(model=selected_model, google_api_key=GOOGLE_API_KEY, temperature=0.2)

rag_prompt_template = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template)

def format_docs(docs):
    return "\n\n".join(doc['text'] for doc in docs)

rag_chain = (
    {"context": lambda x: format_docs(retrieve(x["question"], all_chunks, bge_model, index)), "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

st.set_page_config(layout="wide")

st.title("RAG System Interactive Query")

st.write("Enter your question below to get an answer generated by the RAG system.")

# User input
question = st.text_input("Your Question:", "What are Apple’s goals for carbon neutrality?")

if st.button("Get Answer"):
    if question:
        with st.spinner("Retrieving context and generating answer..."):
            try:
                # Retrieve relevant documents
                retrieved_docs = retrieve(question, all_chunks, bge_model, index, k=3) # Retrieve top 3 documents

                # Generate answer using the RAG chain
                answer = rag_chain.invoke({"question": question})

                st.subheader("Generated Answer:")
                st.write(answer)

                st.subheader("Supporting Documents:")
                for i, doc in enumerate(retrieved_docs):
                    st.write(f"**Document {i+1} (Company: {doc['metadata']['company']}, Page: {doc['metadata']['page']}):**")
                    st.info(doc['text'])
                    st.write("---")
            except Exception as e:
                st.error(f"An error occurred: {e}")
    else:
        st.warning("Please enter a question.")

In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
!streamlit run /content/app.py &>/content/logs.txt & npx localtunnel --port 8501